# 04_step4_verification

Longitudinal causal verification: does achieved BMI reduction (t0->t1) lower observed hypertension new-onset (t1->t2)? Compares crude association, covariate-adjusted logistic, propensity-score matching (ATT), and baseline-BMI stratification (Table 3). Builds the 3-wave Step 4 sample.

In [1]:
# 04_step4_verification.ipynb
# Step 4: verify the counterfactual proposal against observed transitions.
# 3-wave structure: t0 (baseline, disease-free) -> t1 (BMI change) -> t2 (outcome).

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

ROOT = os.path.abspath("..")
DATA = os.path.join(ROOT, "data")
TAB  = os.path.join(ROOT, "results", "tables")

panel = pd.read_parquet(os.path.join(DATA, "khp_panel_long.parquet"))
adult = panel[panel.age >= 19].copy()

TRIPLES = [(2019, 2020, 2021), (2020, 2021, 2022),
           (2021, 2022, 2023), (2022, 2023, 2024)]

rows = []
for t0, t1, t2 in TRIPLES:
    d0 = adult[adult.year == t0][["PIDWON", "BMI", "HTN", "age", "SEX", "smoke_cur", "exer_reg"]]
    d1 = adult[adult.year == t1][["PIDWON", "BMI", "HTN"]]
    d2 = adult[adult.year == t2][["PIDWON", "HTN"]]
    m = (d0.merge(d1, on="PIDWON", suffixes=("_0", "_1"))
           .merge(d2, on="PIDWON").rename(columns={"HTN": "HTN_2"}))
    # at risk: disease-free through t1 so t2 onset is genuinely new
    r = m[(m["HTN_0"] == 0) & (m["HTN_1"] == 0)] \
          .dropna(subset=["BMI_0", "BMI_1", "HTN_2"]).copy()
    r["dBMI"]        = r["BMI_1"] - r["BMI_0"]
    r["incident_t2"] = (r["HTN_2"] == 1).astype(int)
    r["achieved"]    = (r["dBMI"] <= -1.0).astype(int)   # >= 1 kg/m2 reduction
    r["t0y"]         = t0
    rows.append(r)

S = pd.concat(rows, ignore_index=True)
S.to_parquet(os.path.join(DATA, "step4_data.parquet"))
print("Step4 sample:", len(S), "| achieved:", int(S["achieved"].sum()))

crude = S.groupby("achieved")["incident_t2"].mean() * 100
print(f"Crude incidence  achieved={crude[1]:.2f}%  not={crude[0]:.2f}%  "
      f"RR={crude[1]/crude[0]:.3f}")

Step4 sample: 21882 | achieved: 2349
Crude incidence  achieved=3.53%  not=3.05%  RR=1.160


In [2]:
# (1) Covariate-adjusted logistic regression.
S["female"] = (S["SEX"] == 2).astype(int)
S = S.dropna(subset=["smoke_cur", "exer_reg", "BMI_0", "age"])

m = smf.logit(
    "incident_t2 ~ achieved + BMI_0 + age + female + smoke_cur + exer_reg + C(t0y)",
    data=S).fit(disp=0)
OR = np.exp(m.params["achieved"]); ci = np.exp(m.conf_int().loc["achieved"])
print(f"(1) Adjusted OR = {OR:.3f} (95% CI {ci[0]:.3f}-{ci[1]:.3f}), "
      f"p={m.pvalues['achieved']:.3f}")

(1) Adjusted OR = 1.056 (95% CI 0.827-1.347), p=0.663


In [3]:
# (2) Propensity-score matching (1:1, caliper) -> ATT.
covs = ["BMI_0", "age", "female", "smoke_cur", "exer_reg", "t0y"]
Xps  = StandardScaler().fit_transform(S[covs])
S["ps"] = LogisticRegression(max_iter=1000).fit(Xps, S["achieved"]).predict_proba(Xps)[:, 1]

treat = S[S.achieved == 1].copy()
ctrl  = S[S.achieved == 0].copy()
nn = NearestNeighbors(n_neighbors=1).fit(ctrl[["ps"]].values)
dist, idx = nn.kneighbors(treat[["ps"]].values)
caliper = 0.05 * S["ps"].std()
keep = dist.flatten() <= caliper
tm = treat[keep]
cm = ctrl.iloc[idx.flatten()][keep]

att = (tm["incident_t2"].mean() - cm["incident_t2"].mean()) * 100
print(f"(2) PSM matched pairs: {len(tm)}")
print(f"    achieved={tm['incident_t2'].mean()*100:.2f}%  "
      f"not={cm['incident_t2'].mean()*100:.2f}%  ATT={att:+.2f}pp")
for c in ["BMI_0", "age", "smoke_cur"]:
    smd = (tm[c].mean() - cm[c].mean()) / np.sqrt((tm[c].var() + cm[c].var()) / 2)
    print(f"    SMD {c}: {smd:+.3f}")

(2) PSM matched pairs: 2328
    achieved=3.57%  not=3.74%  ATT=-0.17pp
    SMD BMI_0: +0.003
    SMD age: -0.032
    SMD smoke_cur: -0.024


In [4]:
# (3) Baseline-BMI stratification and Table 3 assembly.
def strat(lo, hi):
    sub = S[(S.BMI_0 >= lo) & (S.BMI_0 < hi)]
    g = sub.groupby("achieved")["incident_t2"].mean() * 100
    return g.get(1, np.nan), g.get(0, np.nan)

ov_a, ov_n = strat(25, 100)
print(f"(3) Overweight+ (>=25): achieved={ov_a:.2f}%  not={ov_n:.2f}%")

tab3 = pd.DataFrame({
    "Analysis":        ["Crude", "PSM matched", "Overweight+ stratum"],
    "Achieved_%":      [round(crude[1], 2), round(tm['incident_t2'].mean()*100, 2), round(ov_a, 2)],
    "NotAchieved_%":   [round(crude[0], 2), round(cm['incident_t2'].mean()*100, 2), round(ov_n, 2)],
})
tab3.to_csv(os.path.join(TAB, "table3_step4_results.csv"), index=False)
print(tab3.to_string(index=False))

(3) Overweight+ (>=25): achieved=4.07%  not=3.95%
           Analysis  Achieved_%  NotAchieved_%
              Crude        3.53           3.05
        PSM matched        3.57           3.74
Overweight+ stratum        4.07           3.95
